In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get update -y
!apt-get install -y libegl1-mesa libgl1-mesa-glx libosmesa6 ffmpeg
!pip install -q "gymnasium[mujoco]" mujoco torch numpy matplotlib

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 7,549 B in 1s (6,588 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to pr

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

BASE_DIR = "/content/drive/MyDrive/mujoco_humanoid_sac_jacobian"
VIDEO_DIR = os.path.join(BASE_DIR, "videos")
PLOT_DIR = os.path.join(BASE_DIR, "plots")
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

CONFIG = {
    "env_name": "Humanoid-v5",
    "seed": 42,
    "device": "cuda",

    # ===== [FIX 1] 3M steps (기존 100k → 30배 증가) =====
    "total_steps": 750_000,
    "steps_per_epoch": 50_000,
    "start_steps": 10_000,       # 전체의 0.3%로 줄임
    "update_after": 5_000,
    "update_every": 1,           # 매 스텝마다 업데이트
    "update_steps": 1,           # UTD ratio = 1

    # ===== [FIX 3] 더 큰 네트워크 =====
    "hidden_dims": [512, 512, 512],  # 3-layer 512
    "batch_size": 256,
    "replay_size": 1_000_000,

    "gamma": 0.99,
    "tau": 0.005,
    "actor_lr": 3e-4,
    "critic_lr": 3e-4,
    "alpha_lr": 1e-4,            # alpha lr 약간 낮춤 (안정성)
    "init_alpha": 0.1,           # 초기 alpha 낮춤
    "target_entropy_scale": 1.0,

    "grad_clip": 1.0,
    "normalize_obs": True,

    # imitation learning (기본 OFF — 랜덤 데모는 해로움)
    "use_il": False,
    "demo_path": "/content/drive/MyDrive/mujoco_humanoid_sac_jacobian/demos_humanoid.npz",
    "bc_coef": 0.3,
    "bc_pretrain_steps": 0,

    # regularization — jacobian reg 제거 (학습 초기 방해)
    "jacobian_reg_coef": 0.0,
    "action_l2_coef": 1e-4,      # 가벼운 action L2만 사용

    "eval_episodes": 5,
}


In [ ]:
import math
import copy
import time
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import gymnasium as gym
from gymnasium.wrappers import RecordVideo

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mlp(sizes, activation=nn.ReLU, output_activation=nn.Identity):
    layers = []
    for i in range(len(sizes) - 1):
        act = activation if i < len(sizes) - 2 else output_activation
        layers += [nn.Linear(sizes[i], sizes[i + 1]), act()]
    return nn.Sequential(*layers)


In [ ]:
class RunningMeanStd:
    def __init__(self, shape, eps=1e-4):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = eps

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            x = x[None, :]
        batch_mean = x.mean(axis=0)
        batch_var = x.var(axis=0)
        batch_count = x.shape[0]
        self._update_from_moments(batch_mean, batch_var, batch_count)

    def _update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        total_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / total_count

        m_a = self.var * self.count
        m_b = batch_var * batch_count
        m2 = m_a + m_b + (delta ** 2) * self.count * batch_count / total_count
        new_var = m2 / total_count

        self.mean = new_mean
        self.var = new_var
        self.count = total_count

    def normalize(self, x: np.ndarray, clip=10.0):
        x = (x - self.mean) / np.sqrt(self.var + 1e-8)
        return np.clip(x, -clip, clip).astype(np.float32)

In [ ]:
class ReplayBuffer:
    """
    [FIX 2] Raw obs를 저장하고, 샘플링 시 정규화하는 Replay Buffer.
    기존 코드는 정규화된 obs를 저장해서 RunningMeanStd 통계 변화에 따라
    초기 데이터와 후기 데이터의 스케일이 불일치하는 치명적 버그가 있었음.
    """
    def __init__(self, obs_dim, act_dim, size):
        self.obs = np.zeros((size, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((size, obs_dim), dtype=np.float32)
        self.acts = np.zeros((size, act_dim), dtype=np.float32)
        self.rews = np.zeros((size, 1), dtype=np.float32)
        self.done = np.zeros((size, 1), dtype=np.float32)

        self.ptr = 0
        self.size = 0
        self.max_size = size

    def store(self, obs, act, rew, next_obs, done):
        """Raw (정규화 전) obs/next_obs를 저장"""
        self.obs[self.ptr] = obs
        self.acts[self.ptr] = act
        self.rews[self.ptr] = rew
        self.next_obs[self.ptr] = next_obs
        self.done[self.ptr] = done

        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)

    def sample_batch(self, batch_size, device, obs_rms=None):
        """샘플링 시점의 최신 통계로 정규화"""
        idxs = np.random.randint(0, self.size, size=batch_size)

        obs = self.obs[idxs]
        next_obs = self.next_obs[idxs]

        if obs_rms is not None:
            obs = obs_rms.normalize(obs)
            next_obs = obs_rms.normalize(next_obs)

        return {
            "obs": torch.tensor(obs, dtype=torch.float32, device=device),
            "acts": torch.tensor(self.acts[idxs], dtype=torch.float32, device=device),
            "rews": torch.tensor(self.rews[idxs], dtype=torch.float32, device=device),
            "next_obs": torch.tensor(next_obs, dtype=torch.float32, device=device),
            "done": torch.tensor(self.done[idxs], dtype=torch.float32, device=device),
        }


Imitation Learning용 DemoDataset

In [ ]:
class DemoDataset:
    def __init__(self, demo_path, obs_rms=None):
        data = np.load(demo_path)
        self.obs = data["obs"].astype(np.float32)
        self.acts = data["acts"].astype(np.float32)
        self.size = len(self.obs)
        self.obs_rms = obs_rms
        if self.size == 0:
            raise ValueError("Empty demo dataset")

    def sample(self, batch_size, device):
        idxs = np.random.randint(0, self.size, size=batch_size)
        obs = self.obs[idxs]
        acts = self.acts[idxs]
        if self.obs_rms is not None:
            obs = self.obs_rms.normalize(obs)
        return {
            "obs": torch.tensor(obs, dtype=torch.float32, device=device),
            "acts": torch.tensor(acts, dtype=torch.float32, device=device),
        }

In [ ]:
LOG_STD_MIN = -20
LOG_STD_MAX = 2

class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dims, act_limit):
        """
        [FIX 3] hidden_dims를 리스트로 받아 깊은 네트워크 지원.
        기존: [obs, 256, 256] → 수정: [obs, 512, 512, 512]
        """
        super().__init__()
        sizes = [obs_dim] + list(hidden_dims)
        self.net = mlp(sizes, activation=nn.ReLU, output_activation=nn.ReLU)
        self.mu = nn.Linear(hidden_dims[-1], act_dim)
        self.log_std = nn.Linear(hidden_dims[-1], act_dim)
        self.act_limit = act_limit
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
            nn.init.constant_(m.bias, 0.0)

    def forward(self, obs):
        h = self.net(obs)
        mu = self.mu(h)
        log_std = self.log_std(h)
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX)
        std = torch.exp(log_std)
        return mu, std, log_std

    def sample(self, obs):
        mu, std, log_std = self.forward(obs)
        dist = torch.distributions.Normal(mu, std)
        z = dist.rsample()
        tanh_z = torch.tanh(z)
        action = self.act_limit * tanh_z

        log_prob = dist.log_prob(z).sum(dim=-1, keepdim=True)
        log_prob -= torch.sum(torch.log(1 - tanh_z.pow(2) + 1e-6), dim=-1, keepdim=True)

        mu_action = self.act_limit * torch.tanh(mu)
        return action, log_prob, mu_action, log_std

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        if obs.ndim == 1:
            obs = obs[None, :]
        obs_t = torch.tensor(obs, dtype=torch.float32, device=next(self.parameters()).device)
        if deterministic:
            mu, _, _ = self.forward(obs_t)
            action = self.act_limit * torch.tanh(mu)
        else:
            action, _, _, _ = self.sample(obs_t)
        return action.cpu().numpy()[0]


class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dims):
        super().__init__()
        sizes = [obs_dim + act_dim] + list(hidden_dims) + [1]
        self.q = mlp(sizes, activation=nn.ReLU)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
            nn.init.constant_(m.bias, 0.0)

    def forward(self, obs, act):
        return self.q(torch.cat([obs, act], dim=-1))


In [ ]:
class SACAgent:
    def __init__(self, obs_dim, act_dim, act_limit, cfg):
        self.cfg = cfg
        self.device = cfg["device"]
        self.gamma = cfg["gamma"]
        self.tau = cfg["tau"]
        self.act_limit = act_limit

        hidden_dims = cfg["hidden_dims"]

        self.actor = Actor(obs_dim, act_dim, hidden_dims, act_limit).to(self.device)

        self.q1 = Critic(obs_dim, act_dim, hidden_dims).to(self.device)
        self.q2 = Critic(obs_dim, act_dim, hidden_dims).to(self.device)

        self.q1_target = copy.deepcopy(self.q1).to(self.device)
        self.q2_target = copy.deepcopy(self.q2).to(self.device)

        for p in self.q1_target.parameters():
            p.requires_grad = False
        for p in self.q2_target.parameters():
            p.requires_grad = False

        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=cfg["actor_lr"])
        self.q1_opt = torch.optim.Adam(self.q1.parameters(), lr=cfg["critic_lr"])
        self.q2_opt = torch.optim.Adam(self.q2.parameters(), lr=cfg["critic_lr"])

        self.target_entropy = -float(act_dim) * cfg["target_entropy_scale"]
        self.log_alpha = torch.tensor(
            math.log(cfg["init_alpha"]),
            dtype=torch.float32,
            device=self.device,
            requires_grad=True,
        )
        self.alpha_opt = torch.optim.Adam([self.log_alpha], lr=cfg["alpha_lr"])

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def _jacobian_reg(self, obs):
        obs_req = obs.detach().clone().requires_grad_(True)
        mu, _, _ = self.actor.forward(obs_req)
        scalar = (mu ** 2).sum()
        grad = torch.autograd.grad(
            scalar,
            obs_req,
            create_graph=True,
            retain_graph=True,
        )[0]
        jac_reg = grad.pow(2).mean()
        return jac_reg

    def update(self, batch, demo_batch=None):
        obs = batch["obs"]
        acts = batch["acts"]
        rews = batch["rews"]
        next_obs = batch["next_obs"]
        done = batch["done"]

        # --- Critic update ---
        with torch.no_grad():
            next_action, next_log_prob, _, _ = self.actor.sample(next_obs)
            q1_next = self.q1_target(next_obs, next_action)
            q2_next = self.q2_target(next_obs, next_action)
            q_next = torch.min(q1_next, q2_next) - self.alpha.detach() * next_log_prob
            target_q = rews + self.gamma * (1 - done) * q_next

        q1_pred = self.q1(obs, acts)
        q2_pred = self.q2(obs, acts)

        q1_loss = F.mse_loss(q1_pred, target_q)
        q2_loss = F.mse_loss(q2_pred, target_q)

        self.q1_opt.zero_grad()
        q1_loss.backward()
        if self.cfg["grad_clip"] > 0:
            torch.nn.utils.clip_grad_norm_(self.q1.parameters(), self.cfg["grad_clip"])
        self.q1_opt.step()

        self.q2_opt.zero_grad()
        q2_loss.backward()
        if self.cfg["grad_clip"] > 0:
            torch.nn.utils.clip_grad_norm_(self.q2.parameters(), self.cfg["grad_clip"])
        self.q2_opt.step()

        # --- Actor update ---
        for p in self.q1.parameters():
            p.requires_grad = False
        for p in self.q2.parameters():
            p.requires_grad = False

        new_action, log_prob, _, log_std = self.actor.sample(obs)
        q1_new = self.q1(obs, new_action)
        q2_new = self.q2(obs, new_action)
        q_new = torch.min(q1_new, q2_new)

        sac_actor_loss = (self.alpha.detach() * log_prob - q_new).mean()

        bc_loss = torch.tensor(0.0, device=self.device)
        if demo_batch is not None and self.cfg["use_il"]:
            demo_obs = demo_batch["obs"]
            demo_acts = demo_batch["acts"]
            pred_acts, _, _, _ = self.actor.sample(demo_obs)
            bc_loss = F.mse_loss(pred_acts, demo_acts)

        jac_reg = torch.tensor(0.0, device=self.device)
        if self.cfg["jacobian_reg_coef"] > 0:
            jac_reg = self._jacobian_reg(obs)

        action_l2 = new_action.pow(2).mean()

        actor_loss = (
            sac_actor_loss
            + self.cfg["bc_coef"] * bc_loss
            + self.cfg["jacobian_reg_coef"] * jac_reg
            + self.cfg["action_l2_coef"] * action_l2
        )

        self.actor_opt.zero_grad()
        actor_loss.backward()
        if self.cfg["grad_clip"] > 0:
            actor_grad_norm = torch.nn.utils.clip_grad_norm_(self.actor.parameters(), self.cfg["grad_clip"]).item()
        else:
            actor_grad_norm = 0.0
        self.actor_opt.step()

        for p in self.q1.parameters():
            p.requires_grad = True
        for p in self.q2.parameters():
            p.requires_grad = True

        # --- Alpha update ---
        alpha_loss = -(self.log_alpha * (log_prob.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

        # --- Target network soft update ---
        with torch.no_grad():
            for p, p_targ in zip(self.q1.parameters(), self.q1_target.parameters()):
                p_targ.data.mul_(1 - self.tau)
                p_targ.data.add_(self.tau * p.data)
            for p, p_targ in zip(self.q2.parameters(), self.q2_target.parameters()):
                p_targ.data.mul_(1 - self.tau)
                p_targ.data.add_(self.tau * p.data)

        q_gap = (q1_pred - q2_pred).abs().mean()
        entropy_proxy = (-log_prob).mean()
        action_mean_abs = new_action.abs().mean()
        log_std_mean = log_std.mean()

        return {
            "q1_loss": q1_loss.item(),
            "q2_loss": q2_loss.item(),
            "actor_loss": actor_loss.item(),
            "sac_actor_loss": sac_actor_loss.item(),
            "bc_loss": bc_loss.item(),
            "alpha": self.alpha.item(),
            "entropy_proxy": entropy_proxy.item(),
            "q1_mean": q1_pred.mean().item(),
            "q2_mean": q2_pred.mean().item(),
            "target_q_mean": target_q.mean().item(),
            "q_gap": q_gap.item(),
            "action_mean_abs": action_mean_abs.item(),
            "log_std_mean": log_std_mean.item(),
            "jac_reg": jac_reg.item(),
            "actor_grad_norm": actor_grad_norm,
        }


In [ ]:
@torch.no_grad()
def evaluate(agent, env_name, seed, obs_rms=None, episodes=3):
    env = gym.make(env_name)
    returns = []
    lens = []

    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + 1000 + ep)
        done = False
        ep_ret = 0.0
        ep_len = 0

        while not done:
            obs_in = obs_rms.normalize(obs) if obs_rms is not None else obs.astype(np.float32)
            action = agent.actor.act(obs_in, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_ret += reward
            ep_len += 1

        returns.append(ep_ret)
        lens.append(ep_len)

    env.close()
    return float(np.mean(returns)), float(np.std(returns)), float(np.mean(lens))

In [ ]:
def save_plots(history, plot_dir):
    epochs = np.arange(1, len(history["epoch"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_ep_ret_mean"], label="train")
    plt.plot(epochs, history["eval_return_mean"], label="eval")
    plt.xlabel("epoch")
    plt.ylabel("return")
    plt.title("Reward by epoch")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "reward_by_epoch.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(history["entropy_proxy"], history["eval_return_mean"], alpha=0.8)
    plt.xlabel("entropy proxy (-log pi)")
    plt.ylabel("eval return")
    plt.title("Exploration vs Performance")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "exploration_vs_performance.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(history["alpha"], history["eval_return_mean"], alpha=0.8)
    plt.xlabel("alpha")
    plt.ylabel("eval return")
    plt.title("Alpha vs Performance")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "alpha_vs_performance.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["q1_loss"], label="q1_loss")
    plt.plot(epochs, history["q2_loss"], label="q2_loss")
    plt.plot(epochs, history["actor_loss"], label="actor_loss")
    if max(history["bc_loss"]) > 0:
        plt.plot(epochs, history["bc_loss"], label="bc_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Losses")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "losses.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["q_gap"], label="q_gap")
    plt.xlabel("epoch")
    plt.ylabel("value")
    plt.title("Twin Q Disagreement")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "q_gap.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["action_mean_abs"], label="mean |action|")
    plt.plot(epochs, history["log_std_mean"], label="mean log_std")
    plt.xlabel("epoch")
    plt.ylabel("value")
    plt.title("Action Scale / Variance")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "action_scale_variance.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["entropy_proxy"], label="entropy_proxy")
    plt.plot(epochs, history["alpha"], label="alpha")
    plt.xlabel("epoch")
    plt.ylabel("value")
    plt.title("Exploration Curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "exploration_curves.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["jac_reg"], label="jacobian_reg")
    plt.plot(epochs, history["actor_grad_norm"], label="actor_grad_norm")
    plt.xlabel("epoch")
    plt.ylabel("value")
    plt.title("Jacobian Reg / Grad Norm")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "jacobian_reg.png"))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(history["q_gap"], history["eval_return_mean"], alpha=0.8)
    plt.xlabel("q_gap")
    plt.ylabel("eval_return")
    plt.title("Q-gap vs Eval Return")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "qgap_vs_eval.png"))
    plt.close()

    np.savez(os.path.join(plot_dir, "history.npz"), **{k: np.array(v) for k, v in history.items()})

In [ ]:
def train(cfg):
    set_seed(cfg["seed"])

    env = gym.make(cfg["env_name"])
    obs, _ = env.reset(seed=cfg["seed"])
    env.action_space.seed(cfg["seed"])

    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    act_limit = float(env.action_space.high[0])

    print("device:", cfg["device"])
    print("obs_dim:", obs_dim)
    print("act_dim:", act_dim)
    print("act_limit:", act_limit)
    print("hidden_dims:", cfg["hidden_dims"])
    print("total_steps:", cfg["total_steps"])

    obs_rms = RunningMeanStd(shape=(obs_dim,)) if cfg["normalize_obs"] else None
    if obs_rms is not None:
        obs_rms.update(obs)

    agent = SACAgent(obs_dim, act_dim, act_limit, cfg)
    replay = ReplayBuffer(obs_dim, act_dim, cfg["replay_size"])

    # 파라미터 수 출력
    actor_params = sum(p.numel() for p in agent.actor.parameters())
    critic_params = sum(p.numel() for p in agent.q1.parameters())
    print(f"actor params: {actor_params:,}")
    print(f"critic params (each): {critic_params:,}")

    demo_dataset = None
    if cfg["use_il"]:
        demo_dataset = DemoDataset(cfg["demo_path"], obs_rms=obs_rms)
        print(f"[IL] loaded {demo_dataset.size} demo samples")

    if cfg["use_il"] and cfg["bc_pretrain_steps"] > 0:
        print(f"[IL] BC pretrain {cfg['bc_pretrain_steps']} steps")
        for step in range(cfg["bc_pretrain_steps"]):
            batch = demo_dataset.sample(cfg["batch_size"], cfg["device"])
            pred, _, _, _ = agent.actor.sample(batch["obs"])
            bc_loss = F.mse_loss(pred, batch["acts"])

            agent.actor_opt.zero_grad()
            bc_loss.backward()
            if cfg["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), cfg["grad_clip"])
            agent.actor_opt.step()

            if (step + 1) % 1000 == 0:
                print(f"[BC pretrain] step={step+1}, bc_loss={bc_loss.item():.6f}")

    history = {
        "epoch": [],
        "train_ep_ret_mean": [],
        "train_ep_len_mean": [],
        "eval_return_mean": [],
        "eval_return_std": [],
        "eval_ep_len_mean": [],
        "q1_loss": [],
        "q2_loss": [],
        "actor_loss": [],
        "bc_loss": [],
        "alpha": [],
        "entropy_proxy": [],
        "q1_mean": [],
        "q2_mean": [],
        "target_q_mean": [],
        "q_gap": [],
        "action_mean_abs": [],
        "log_std_mean": [],
        "jac_reg": [],
        "actor_grad_norm": [],
    }

    obs, _ = env.reset(seed=cfg["seed"])
    train_ep_returns = []
    train_ep_lens = []
    ep_ret = 0.0
    ep_len = 0

    num_epochs = math.ceil(cfg["total_steps"] / cfg["steps_per_epoch"])
    global_step = 0
    last_metrics = None
    best_eval = -float("inf")
    t0 = time.time()

    for epoch in range(1, num_epochs + 1):
        for _ in range(cfg["steps_per_epoch"]):
            global_step += 1

            # [FIX 2] obs_rms 업데이트는 하되, replay에는 RAW obs 저장
            if obs_rms is not None:
                obs_rms.update(obs[None, :])
                obs_input = obs_rms.normalize(obs)
            else:
                obs_input = obs.astype(np.float32)

            if global_step < cfg["start_steps"]:
                action = env.action_space.sample()
            else:
                action = agent.actor.act(obs_input, deterministic=False)

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # [FIX 2] RAW obs를 저장! (정규화된 obs_input이 아님)
            replay.store(obs, action, reward, next_obs, float(terminated))
            # Note: done 대신 terminated만 저장 (truncation은 실제 terminal이 아님)

            obs = next_obs
            ep_ret += reward
            ep_len += 1

            if done:
                train_ep_returns.append(ep_ret)
                train_ep_lens.append(ep_len)
                obs, _ = env.reset()
                ep_ret = 0.0
                ep_len = 0

            if global_step >= cfg["update_after"] and global_step % cfg["update_every"] == 0 and replay.size >= cfg["batch_size"]:
                for _ in range(cfg["update_steps"]):
                    # [FIX 2] 샘플링 시 최신 obs_rms로 정규화
                    rl_batch = replay.sample_batch(cfg["batch_size"], cfg["device"], obs_rms=obs_rms)
                    demo_batch = demo_dataset.sample(cfg["batch_size"], cfg["device"]) if cfg["use_il"] else None
                    last_metrics = agent.update(rl_batch, demo_batch)

            if global_step >= cfg["total_steps"]:
                break

        eval_mean, eval_std, eval_len = evaluate(
            agent,
            cfg["env_name"],
            cfg["seed"],
            obs_rms=obs_rms,
            episodes=cfg["eval_episodes"],
        )

        if len(train_ep_returns) > 0:
            train_ret_mean = float(np.mean(train_ep_returns))
            train_len_mean = float(np.mean(train_ep_lens))
        else:
            train_ret_mean = float("nan")
            train_len_mean = float("nan")

        if last_metrics is None:
            last_metrics = {
                "q1_loss": np.nan,
                "q2_loss": np.nan,
                "actor_loss": np.nan,
                "bc_loss": 0.0,
                "alpha": np.nan,
                "entropy_proxy": np.nan,
                "q1_mean": np.nan,
                "q2_mean": np.nan,
                "target_q_mean": np.nan,
                "q_gap": np.nan,
                "action_mean_abs": np.nan,
                "log_std_mean": np.nan,
                "jac_reg": 0.0,
                "actor_grad_norm": np.nan,
            }

        history["epoch"].append(epoch)
        history["train_ep_ret_mean"].append(train_ret_mean)
        history["train_ep_len_mean"].append(train_len_mean)
        history["eval_return_mean"].append(eval_mean)
        history["eval_return_std"].append(eval_std)
        history["eval_ep_len_mean"].append(eval_len)
        history["q1_loss"].append(last_metrics["q1_loss"])
        history["q2_loss"].append(last_metrics["q2_loss"])
        history["actor_loss"].append(last_metrics["actor_loss"])
        history["bc_loss"].append(last_metrics["bc_loss"])
        history["alpha"].append(last_metrics["alpha"])
        history["entropy_proxy"].append(last_metrics["entropy_proxy"])
        history["q1_mean"].append(last_metrics["q1_mean"])
        history["q2_mean"].append(last_metrics["q2_mean"])
        history["target_q_mean"].append(last_metrics["target_q_mean"])
        history["q_gap"].append(last_metrics["q_gap"])
        history["action_mean_abs"].append(last_metrics["action_mean_abs"])
        history["log_std_mean"].append(last_metrics["log_std_mean"])
        history["jac_reg"].append(last_metrics["jac_reg"])
        history["actor_grad_norm"].append(last_metrics["actor_grad_norm"])

        elapsed = (time.time() - t0) / 60
        eta = elapsed / epoch * (num_epochs - epoch)

        print(
            f"[Epoch {epoch}/{num_epochs}] "
            f"step={global_step} "
            f"train_ret={train_ret_mean:.1f} "
            f"eval_ret={eval_mean:.1f}±{eval_std:.1f} "
            f"alpha={last_metrics['alpha']:.4f} "
            f"entropy={last_metrics['entropy_proxy']:.2f} "
            f"qgap={last_metrics['q_gap']:.2f} "
            f"time={elapsed:.1f}m ETA={eta:.1f}m"
        )

        save_plots(history, PLOT_DIR)

        # Best model 저장
        if eval_mean > best_eval:
            best_eval = eval_mean
            ckpt_best = {
                "actor": agent.actor.state_dict(),
                "q1": agent.q1.state_dict(),
                "q2": agent.q2.state_dict(),
                "q1_target": agent.q1_target.state_dict(),
                "q2_target": agent.q2_target.state_dict(),
                "log_alpha": agent.log_alpha.detach().cpu().numpy(),
                "cfg": cfg,
                "obs_rms_mean": obs_rms.mean if obs_rms is not None else None,
                "obs_rms_var": obs_rms.var if obs_rms is not None else None,
                "obs_rms_count": obs_rms.count if obs_rms is not None else None,
                "history": history,
                "best_eval": best_eval,
            }
            torch.save(ckpt_best, os.path.join(CKPT_DIR, "humanoid_sac_best.pt"))
            print(f"  >> New best eval: {best_eval:.1f}")

        # Latest checkpoint 항상 저장
        ckpt = {
            "actor": agent.actor.state_dict(),
            "q1": agent.q1.state_dict(),
            "q2": agent.q2.state_dict(),
            "q1_target": agent.q1_target.state_dict(),
            "q2_target": agent.q2_target.state_dict(),
            "log_alpha": agent.log_alpha.detach().cpu().numpy(),
            "cfg": cfg,
            "obs_rms_mean": obs_rms.mean if obs_rms is not None else None,
            "obs_rms_var": obs_rms.var if obs_rms is not None else None,
            "obs_rms_count": obs_rms.count if obs_rms is not None else None,
            "history": history,
        }
        torch.save(ckpt, os.path.join(CKPT_DIR, "humanoid_sac_latest.pt"))

        with open(os.path.join(BASE_DIR, "config.json"), "w") as f:
            json.dump(cfg, f, indent=2)

        train_ep_returns.clear()
        train_ep_lens.clear()

        if global_step >= cfg["total_steps"]:
            break

    env.close()
    print(f"done. total time: {(time.time() - t0)/60:.1f} min, best eval: {best_eval:.1f}")
    return agent, obs_rms, history


In [ ]:
agent, obs_rms, history = train(CONFIG)

device: cuda
obs_dim: 348
act_dim: 17
act_limit: 0.4000000059604645
hidden_dims: [512, 512, 512]
total_steps: 750000
actor params: 721,442
critic params (each): 713,217
[Epoch 1/15] step=50000 train_ret=263.0 eval_ret=602.6±154.9 alpha=0.0824 entropy=-15.94 qgap=6.98 time=12.1m ETA=169.9m
  >> New best eval: 602.6
[Epoch 2/15] step=100000 train_ret=530.5 eval_ret=634.0±138.5 alpha=0.1615 entropy=-18.81 qgap=18.53 time=25.8m ETA=167.6m
  >> New best eval: 634.0
[Epoch 3/15] step=150000 train_ret=563.3 eval_ret=539.4±131.4 alpha=0.5654 entropy=-18.02 qgap=80.37 time=39.3m ETA=157.0m
[Epoch 4/15] step=200000 train_ret=545.6 eval_ret=753.5±66.1 alpha=1.9616 entropy=-19.42 qgap=343.43 time=52.7m ETA=145.0m
  >> New best eval: 753.5
[Epoch 5/15] step=250000 train_ret=620.0 eval_ret=472.5±335.3 alpha=1.4297 entropy=-17.92 qgap=241.90 time=66.2m ETA=132.4m
[Epoch 6/15] step=300000 train_ret=652.0 eval_ret=974.0±364.1 alpha=0.2303 entropy=-16.84 qgap=45.79 time=79.7m ETA=119.5m
  >> New best ev

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

import torch
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

# best 모델 사용
ckpt_path = os.path.join(CKPT_DIR, "humanoid_sac_best.pt")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(CKPT_DIR, "humanoid_sac_latest.pt")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

env = gym.make("Humanoid-v5", render_mode="rgb_array")
env = RecordVideo(env, video_folder=VIDEO_DIR, episode_trigger=lambda e: True)

obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
act_limit = float(env.action_space.high[0])

hidden_dims = ckpt["cfg"].get("hidden_dims", [256, 256])
actor = Actor(obs_dim, act_dim, hidden_dims, act_limit)
actor.load_state_dict(ckpt["actor"])
actor.eval()

obs_rms_mean = ckpt["obs_rms_mean"]
obs_rms_var = ckpt["obs_rms_var"]

obs, _ = env.reset(seed=0)
ep_ret = 0.0

terminated = truncated = False
while not (terminated or truncated):
    if obs_rms_mean is not None:
        obs_in = (obs - obs_rms_mean) / np.sqrt(obs_rms_var + 1e-8)
        obs_in = np.clip(obs_in, -10.0, 10.0).astype(np.float32)
    else:
        obs_in = obs.astype(np.float32)

    with torch.no_grad():
        obs_t = torch.tensor(obs_in, dtype=torch.float32).unsqueeze(0)
        mu, _, _ = actor(obs_t)
        action = act_limit * torch.tanh(mu)
        action = action.squeeze(0).cpu().numpy()

    obs, reward, terminated, truncated, _ = env.step(action)
    ep_ret += reward

env.close()
print("episode return:", ep_ret)


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/mujoco_humanoid_sac_jacobian/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


episode return: 7199.24215902668


In [ ]:
from IPython.display import Video
import os

video_files = sorted([
    os.path.join(VIDEO_DIR, f)
    for f in os.listdir(VIDEO_DIR)
    if f.endswith(".mp4")
])

print(video_files[-1])
Video(video_files[-1], embed=True)

/content/drive/MyDrive/mujoco_humanoid_sac_jacobian/videos/rl-video-episode-0.mp4


imitation learning용 demo 데이터 예시 생성

In [ ]:
import numpy as np

# Humanoid-v5 default dims: obs 348, act 17
obs = np.random.randn(5000, 348).astype(np.float32)
acts = np.random.uniform(low=-0.4, high=0.4, size=(5000, 17)).astype(np.float32)

demo_path = "/content/drive/MyDrive/mujoco_humanoid_sac_jacobian/demos_humanoid.npz"
np.savez(demo_path, obs=obs, acts=acts)
print("saved:", demo_path)

saved: /content/drive/MyDrive/mujoco_humanoid_sac_jacobian/demos_humanoid.npz


IL 켜고 다시 학습하고 싶을 때 config 수정

In [ ]:
CONFIG["use_il"] = True
CONFIG["demo_path"] = "/content/drive/MyDrive/mujoco_humanoid_sac_jacobian/demos_humanoid.npz"
CONFIG["bc_coef"] = 0.3
CONFIG["bc_pretrain_steps"] = 1000
CONFIG["jacobian_reg_coef"] = 1e-4

In [ ]:
agent, obs_rms, history = train(CONFIG)

device: cuda
obs_dim: 348
act_dim: 17
act_limit: 0.4000000059604645
hidden_dims: [512, 512, 512]
total_steps: 750000
actor params: 721,442
critic params (each): 713,217
[IL] loaded 5000 demo samples
[IL] BC pretrain 1000 steps
[BC pretrain] step=1000, bc_loss=0.059697


In [ ]:
hist = np.load(os.path.join(PLOT_DIR, "history.npz"))
for k in hist.files:
    print(k, hist[k].shape)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(hist["q_gap"], hist["eval_return_mean"], alpha=0.8)
plt.xlabel("q_gap")
plt.ylabel("eval_return_mean")
plt.title("Q-gap vs Eval Return")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(hist["jac_reg"], hist["eval_return_mean"], alpha=0.8)
plt.xlabel("jacobian_reg")
plt.ylabel("eval_return_mean")
plt.title("Jacobian Reg vs Eval Return")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(hist["actor_grad_norm"], hist["actor_loss"], alpha=0.8)
plt.xlabel("actor_grad_norm")
plt.ylabel("actor_loss")
plt.title("Actor Grad Norm vs Actor Loss")
plt.tight_layout()
plt.show()